# Class 7: Intent classification, a classic short-text classification task

## Building an airline reservation system using the classic NLP pipeline

A classic approach to building a basic application that can parse reservation requests involves two stages

1. __Intent classification__: A user utterance is classified as one of *N* intents. An intent captures the task that the user would like completed. Given the utterance *I need a flight from New York to Boston*  the intent would be something like *book_flight*

2. __Slot-filling__. Once the system has determnined the user intent it retrieves a *template* associated with that intent. Each template has __slots__ associated with key pieces of information needed to
complete the task.

      
       FLIGHT_TEMPLATE:
             DEPARTURE_CITY: [], ARRIVAL_CITY; []


      I need a flight from __New York__ to __Boston__


       FLIGHT_TEMPLATE:
             DEPARTURE_CITY: ["New York"], ARRIVAL_CITY; ["Boston"]


3. __Create SQL query to write reservation__. The final step often involves using the information from the filled template to create a SQL query. The information can then be written to a database.

        INSERT INTO reservations (departure_city, arrival_city) VALUES ("New York", "Boston")

The goal of this notebook isn't to build a robust, fucntioning reservation system. We're mainly interested in illusrating the basics of the classic, two-stage approach to this task using two fine-tuned BERT model



1.  **Intent Classification**: We'll download an already-finetuned BERT for this. If you have additional intents for your use-case, you'd have to finetune from scratch.  The intent classifier's job is to classify the user's overarching goal or intention. For example, if a user types "I want to book a flight from London to New York next month," the BERT model will classify the intent as `book_flight`.
2.  **Slot Filling**: Additionally, we'll use an already-fine-tuned BERT for the slot-fillign task.  To perform this task, we'll utilize a BERT fine-tuned for the task of named entity recognition (NER) (more on this topic next week). The NER model will extract specific pieces of information, known as "slots," from the user's utterance. In the example above, the slots would be `departure_city: London`, `arrival_city: New York`, and `date: next month`.

Note that we're not implementing the additional step of generating a SQL query and writing to database.



## Dataset


In [ ]:
import pandas as pd

In [ ]:
try:
    import transformers
    print("transformers library is already installed.")
except ImportError:
    print("transformers library not found. Installing...")
    !pip install transformers
    print("transformers library installed.")

## Load Pre-trained BERT Model for Intent Classification

There are many already-finetuned models on HuggingFace. Carefully examine the model card and description to get a sense of the the finetuning task and the target classes.

Here, I've chosen `yeniguno/bert-uncased-intent-classification` to classifythe user's intent from their input query, which is described as follows:


  *"This is a fine-tuned BERT-based model for intent classification, capable of categorizing intents into 82 distinct labels. It was trained on a consolidated dataset of multilingual intent datasets."*



In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "yeniguno/bert-uncased-intent-classification"

# Load the tokenizer
intent_tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model for sequence classification
intent_model = AutoModelForSequenceClassification.from_pretrained(model_name)

print(f"BERT model and tokenizer for intent classification ('{model_name}') loaded successfully.")

**Check target classes**

Most general purpose intent classification models have ```make_reservation``` for ```book_flight``` in their inventory of target classes. Let's make sure that's the case here.




In [ ]:
label_list = intent_model.config.id2label
intent_labels = {i: label for i, label in label_list.items()}

print("Intent labels defined:")
print(intent_labels)

## Load Pre-trained BERT Model for Slot Filling (Named Entity Recognition)

. We will use `dslim/bert-base-NER` to extract specific entities (slots) like 'departure_city', 'arrival_city', and 'date' from the user's utterance.


In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer

slot_filling_model_name = "dslim/bert-base-NER"

# Load the tokenizer for slot filling
slot_filling_tokenizer = AutoTokenizer.from_pretrained(slot_filling_model_name)

# Load the model for token classification (slot filling)
slot_filling_model = AutoModelForTokenClassification.from_pretrained(slot_filling_model_name)

print(f"BERT model and tokenizer for slot filling ('{slot_filling_model_name}') loaded successfully.")

To interpret the model's token classification predictions, we'll need to define the slot labels (entities) that the model is trained to identify. This step will extract these labels from the loaded model's configuration.



In [ ]:
slot_labels = slot_filling_model.config.id2label

print("Slot labels for the NER model:")
print(slot_labels)

## Implement the two-stage logic to first identify user intent and then slot-fill




The `classify_intent` function will take user input, tokenize it using the pre-loaded `intent_tokenizer`, pass it through the `intent_model` to get predictions, and then map the highest probability prediction to its human-readable intent label using the `intent_labels` dictionary.



In [ ]:
import torch
import torch.nn.functional as F

def classify_intent(text):
    """
    Classifies the intent of a given text using the pre-trained BERT model.
    """
    # Tokenize the input text
    inputs = intent_tokenizer(text, return_tensors='pt', truncation=True, padding=True)

    # Get model predictions
    with torch.no_grad():
        outputs = intent_model(**inputs)

    # Apply softmax to logits to get probabilities
    logits = outputs.logits
    probabilities = F.softmax(logits, dim=-1)

    # Get the predicted intent index
    predicted_intent_idx = torch.argmax(probabilities, dim=-1).item()

    # Map the index to the human-readable label
    predicted_intent = intent_labels[predicted_intent_idx]

    return predicted_intent

print("Function 'classify_intent' defined.")

The next step is to define the `extract_slots` function. This function will take a user utterance, tokenize it using the `slot_filling_tokenizer`, get predictions from the `slot_filling_model`, and then extract and categorize entities based on the predicted NER tags, including heuristics for 'departure_city' and 'arrival_city'.

Question: Why can't we extract perferred departure and arrival times?



In [ ]:
def extract_slots(text):
    """
    Extracts slots (entities) from a given text using the pre-trained NER model.
    """
    # Tokenize the input text with offset mapping to link tokens back to original text
    inputs = slot_filling_tokenizer(text, return_tensors='pt', truncation=True, padding=True, return_offsets_mapping=True)
    offsets = inputs.pop('offset_mapping').squeeze().tolist()

    # Get model predictions
    with torch.no_grad():
        outputs = slot_filling_model(**inputs)

    # Get predicted token labels
    predictions = torch.argmax(outputs.logits, dim=-1).squeeze().tolist()
    tokens = slot_filling_tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze().tolist())

    extracted_entities = {}
    current_entity_type = None
    current_entity_value = []
    current_entity_start = -1

    for i, (token_id, pred_id) in enumerate(zip(inputs['input_ids'].squeeze().tolist(), predictions)):
        if token_id in slot_filling_tokenizer.all_special_ids:
            # Skip special tokens ([CLS], [SEP], [PAD])
            if current_entity_type and current_entity_value:
                # Save the last entity if it was being collected
                if 'LOC' in current_entity_type:
                    # Basic heuristic: Check keywords like 'from' or 'to'
                    segment = text[current_entity_start:offsets[i-1][1]]
                    if "from" in text.lower().split(segment.lower())[0].split()[-3:]:
                        extracted_entities['departure_city'] = segment.strip()
                    elif "to" in text.lower().split(segment.lower())[0].split()[-3:]:
                        extracted_entities['arrival_city'] = segment.strip()
                    else:
                        # If no 'from'/'to' context, just label as a generic location
                        extracted_entities[current_entity_type.replace('B-','').replace('I-','').lower()] = segment.strip()

                else:
                    extracted_entities[current_entity_type.replace('B-','').replace('I-','').lower()] = " ".join(current_entity_value).replace(' ##','')
            current_entity_type = None
            current_entity_value = []
            current_entity_start = -1
            continue

        label = slot_labels[pred_id]
        word_start, word_end = offsets[i]

        if label.startswith('B-'):
            if current_entity_type and current_entity_value:
                # Save previous entity before starting a new one
                segment = text[current_entity_start:offsets[i-1][1]]
                if 'LOC' in current_entity_type:
                    if "from" in text.lower().split(segment.lower())[0].split()[-3:]:
                        extracted_entities['departure_city'] = segment.strip()
                    elif "to" in text.lower().split(segment.lower())[0].split()[-3:]:
                        extracted_entities['arrival_city'] = segment.strip()
                    else:
                        extracted_entities[current_entity_type.replace('B-','').replace('I-','').lower()] = segment.strip()
                else:
                    extracted_entities[current_entity_type.replace('B-','').replace('I-','').lower()] = segment.strip()

            current_entity_type = label
            current_entity_value = [text[word_start:word_end]]
            current_entity_start = word_start

        elif label.startswith('I-') and current_entity_type and current_entity_type.endswith(label.split('-')[1]):
            current_entity_value.append(text[word_start:word_end])

        else: # 'O' or mismatching 'I-' tag
            if current_entity_type and current_entity_value:
                segment = text[current_entity_start:offsets[i-1][1]]
                if 'LOC' in current_entity_type:
                    if "from" in text.lower().split(segment.lower())[0].split()[-3:]:
                        extracted_entities['departure_city'] = segment.strip()
                    elif "to" in text.lower().split(segment.lower())[0].split()[-3:]:
                        extracted_entities['arrival_city'] = segment.strip()
                    else:
                        extracted_entities[current_entity_type.replace('B-','').replace('I-','').lower()] = segment.strip()
                else:
                    extracted_entities[current_entity_type.replace('B-','').replace('I-','').lower()] = segment.strip()
            current_entity_type = None
            current_entity_value = []
            current_entity_start = -1

    # Capture any entity remaining at the end of the sentence
    if current_entity_type and current_entity_value:
        segment = text[current_entity_start:offsets[len(predictions)-1][1]]
        if 'LOC' in current_entity_type:
            if "from" in text.lower().split(segment.lower())[0].split()[-3:]:
                extracted_entities['departure_city'] = segment.strip()
            elif "to" in text.lower().split(segment.lower())[0].split()[-3:]:
                extracted_entities['arrival_city'] = segment.strip()
            else:
                extracted_entities[current_entity_type.replace('B-','').replace('I-','').lower()] = segment.strip()
        else:
            extracted_entities[current_entity_type.replace('B-','').replace('I-','').lower()] = segment.strip()


    return extracted_entities

print("Function 'extract_slots' defined.")

The `process_user_query` function integrates the `classify_intent` and `extract_slots` functions to process a user query, determine its intent, extract relevant entities, and print the results.



In [ ]:
def process_user_query(user_query):
    """
    Processes a user query by classifying its intent and extracting relevant slots.
    """
    # Classify intent
    intent = classify_intent(user_query)
    print(f"Detected Intent: {intent}")

    # Extract slots
    slots = extract_slots(user_query)
    print(f"Extracted Slots: {slots}")

    return intent, slots

print("Function 'process_user_query' defined.")

## Demonstrate usage
Let's test the `process_user_query` function with at least three new, representative user queries that were not part of the `df_dataset`. We can quickly see that the model can only handle a small subset of user utterances.



In [ ]:
print("\n--- Testing End-to-End Chatbot Functionality ---")

# Test with new user queries
queries_to_test = [
    "I need to book a flight to Tokyo from London for next Tuesday.",
    "I want to take a plane from Atlanta to New York"
    "What is the status of my flight AC123?",
    "How much luggage can I bring on my trip?"
]

for query in queries_to_test:
    print(f"\nProcessing User Query: '{query}'")
    intent, slots = process_user_query(query)
    print("----------------------------------")


## Testing for robust NLU

Let's see if the system can handle utterances that require world knowledge or pragmatic inference to correctly determine intent and extract slots.




These queries are designed to test the models' ability to handle situations requiring specific world knowledge, subtle distinctions, or pragmatic inference, which are often difficult for models not fine-tuned on a domain-specific dataset.

1.  **World Knowledge (Defunct Airline)**: "I need a TWA reservation for next Sunday." (TWA is a defunct airline).
2.  **Implicit Context/Ambiguity**: "What's the best time to fly to Bali tomorrow morning?" (Requires understanding 'best time' pragmatically, and 'tomorrow morning' as a date/time).
3.  **Specific Airport Codes**: "Book me a one-way ticket from London Gatwick to JFK in New York." (Tests recognition of airport codes and 'one-way' as a flight type).
4.  **Implied Flight Details**: "Is my flight from Berlin to Rome still on time?" (Assumes context of 'my flight' and checks for status, without a flight number).
5.  **Domain-Specific Attributes**: "How many bags are included in a business class ticket?" (Tests understanding 'business class' as a relevant attribute for baggage inquiries).
6.  **Complex Temporal Reference**: "I want to fly first class from LA to Paris in two weeks." (Requires calculation for 'in two weeks' and recognizing 'first class').
7.  **General Inquiry / Ambiguous Terms**: "Could you tell me if there are any cheap flights to Sydney next month?" (Ambiguous 'cheap flights' and general inquiry intent).


In [ ]:
print("\n--- Testing Specific Edge Cases ---")

edge_case_queries = [
    "I need a TWA reservation for next Sunday.",
    "What's the best time to fly to Bali tomorrow morning?",
    "Book me a one-way ticket from London Gatwick to JFK in New York.",
    "Is my flight from Berlin to Rome still on time?",
    "How many bags are included in a business class ticket?",
    "I want to fly first class from LA to Paris in two weeks.",
    "Could you tell me if there are any cheap flights to Sydney next month?"
]

for query in edge_case_queries:
    print(f"\nProcessing Edge Case Query: '{query}'")
    intent, slots = process_user_query(query)
    print("----------------------------------")
